In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

class RBM(tf.keras.Model):
    def __init__(self, visible_units, hidden_units, learning_rate=0.01, k=1):
        super(RBM, self).__init__()
        self.visible_units = visible_units
        self.hidden_units = hidden_units
        self.learning_rate = learning_rate
        self.k = k
        
        self.W = self.add_weight(
            shape=(visible_units, hidden_units),
            initializer=tf.random_normal_initializer(stddev=0.1),
            name='weights'
        )
        self.hb = self.add_weight(
            shape=(hidden_units,), 
            initializer=tf.zeros_initializer(), 
            name='hidden_bias'
        )
        self.vb = self.add_weight(
            shape=(visible_units,), 
            initializer=tf.zeros_initializer(),
            name='visible_bias'
        )

    def visible_to_hidden(self, v):
        h_prob = tf.nn.sigmoid(tf.matmul(v, self.W) + self.hb)
        return h_prob, tf.nn.relu(tf.sign(h_prob - tf.random.uniform(h_prob.shape)))

    def hidden_to_visible(self, h):
        v_prob = tf.nn.sigmoid(tf.matmul(h, self.W, transpose_b=True) + self.vb)
        return v_prob, tf.nn.relu(tf.sign(v_prob - tf.random.uniform(v_prob.shape)))

    def contrastive_divergence(self, v):
        h_prob, h_sample = self.visible_to_hidden(v)
        v_prob, v_sample = self.hidden_to_visible(h_prob)

        for _ in range(self.k - 1):
            h_prob, h_sample = self.visible_to_hidden(v_sample)
            v_prob, v_sample = self.hidden_to_visible(h_prob)

        return v, v_sample, h_prob, h_sample

    def compute_loss(self, v, v_sample, h_prob, h_sample):
        cross_entropy = -tf.reduce_mean(tf.reduce_sum(v * tf.math.log(v_prob + 1e-10) + (1 - v) * tf.math.log(1 - v_prob + 1e-10), axis=1))
        return cross_entropy

    def call(self, v):
        h_prob, _ = self.visible_to_hidden(v)
        return h_prob

    def get_features(self, v):
        return self(v)
        

I0000 00:00:1793595628.674782   37636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793595628.988157   37636 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793595630.292032   37636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [13]:
def train_rbm(rbm, data, epochs=50, batch_size=32):
    optimizer = Adam(learning_rate=rbm.learning_rate)
    trainable_vars = [rbm.W, rbm.hb, rbm.vb]
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for i in range(0, len(data), batch_size):
            batch = data[i:i+batch_size]
            batch = tf.convert_to_tensor(batch, dtype=tf.float32)

            with tf.GradientTape() as tape:
                v, v_sample, h_prob, h_sample = rbm.contrastive_divergence(batch)
                v_prob = tf.nn.sigmoid(tf.matmul(h_prob, rbm.W, transpose_b=True) + rbm.vb)
                loss = -tf.reduce_mean(tf.reduce_sum(batch * tf.math.log(v_prob + 1e-10) + (1 - batch) * tf.math.log(1 - v_prob + 1e-10), axis=1))

            gradients = tape.gradient(loss, rbm.trainable_variables)
            optimizer.apply_gradients(zip(gradients, trainable_vars))
            epoch_loss += loss.numpy()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/(len(data)/batch_size):.4f}")

In [3]:
def extract_features(rbm, data):
    return rbm.get_features(tf.convert_to_tensor(data, dtype=tf.float32)).numpy()

In [4]:
def save_rbm(rbm, path):
    rbm.save_weights(path)

In [5]:
def load_rbm(visible_units, hidden_units, path):
    rbm = RBM(visible_units, hidden_units)
    rbm.load_weights(path)
    return rbm

In [6]:
import numpy as np
import pandas as pd

VOCAB_SIZE = 1001

X_raw = np.load('../data/processed/X_raw.npy')
y_target = np.load('../data/processed/y_target.npy')

print(X_raw.shape, y_target.shape)

def multi_hot_to_indices(multi_hot_vector):
    vec = np.atleast_1d(multi_hot_vector)
    return [i for i in range(VOCAB_SIZE) if i < len(vec) and vec[i] > 0]

(95072, 1001) (14074, 1001)


In [7]:
user_sequences = pd.DataFrame()
user_sequences['MULTI_HOT_SEQ'] = [[basket for basket in seq if np.any(basket)] for seq in X_raw]
print(user_sequences.shape)
user_sequences['BASKET_SEQUENCE'] = user_sequences['MULTI_HOT_SEQ'].apply(
    lambda seq: [multi_hot_to_indices(basket) for basket in seq]
)

(95072, 1)


In [14]:
HIDDEN_UNITS = 500

rbm = RBM(visible_units=VOCAB_SIZE, hidden_units=HIDDEN_UNITS, learning_rate=0.01, k=1)

train_rbm(rbm, X_raw.reshape(-1, VOCAB_SIZE), epochs=50, batch_size=32)

Epoch 1/50, Loss: 5.3911
Epoch 2/50, Loss: 0.0785
Epoch 3/50, Loss: 0.1144
Epoch 4/50, Loss: 0.0823
Epoch 5/50, Loss: 0.0702
Epoch 6/50, Loss: 0.0565
Epoch 7/50, Loss: 0.0548
Epoch 8/50, Loss: 0.0476
Epoch 9/50, Loss: 0.0466
Epoch 10/50, Loss: 0.0417
Epoch 11/50, Loss: 0.0357
Epoch 12/50, Loss: 0.0368
Epoch 13/50, Loss: 0.0317
Epoch 14/50, Loss: 0.0291
Epoch 15/50, Loss: 0.0311
Epoch 16/50, Loss: 0.0269
Epoch 17/50, Loss: 0.0295
Epoch 18/50, Loss: 0.0257
Epoch 19/50, Loss: 0.0249
Epoch 20/50, Loss: 0.0227
Epoch 21/50, Loss: 0.0207
Epoch 22/50, Loss: 0.0206
Epoch 23/50, Loss: 0.0212
Epoch 24/50, Loss: 0.0180
Epoch 25/50, Loss: 0.0190
Epoch 26/50, Loss: 0.0158
Epoch 27/50, Loss: 0.0167
Epoch 28/50, Loss: 0.0170
Epoch 29/50, Loss: 0.0182
Epoch 30/50, Loss: 0.0184
Epoch 31/50, Loss: 0.0160
Epoch 32/50, Loss: 0.0131
Epoch 33/50, Loss: 0.0118
Epoch 34/50, Loss: 0.0138
Epoch 35/50, Loss: 0.0151
Epoch 36/50, Loss: 0.0146
Epoch 37/50, Loss: 0.0121
Epoch 38/50, Loss: 0.0126
Epoch 39/50, Loss: 0.

In [16]:
features = extract_features(rbm, X_raw.reshape(-1, VOCAB_SIZE))

save_rbm(rbm, '../data/rbm/rbm.weights.h5')